In [1]:
import random

class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.left = None
        self.right = None

        self.rank = 0
        while random.random() < 0.5:
            self.rank += 1

    def priority_less(self, other):
        if self.rank != other.rank:
            return self.rank < other.rank
        return self.key > other.key 

class ZipTreeTensor:
    def __init__(self, dims):
        self.root = None
        self.dims = dims

    def _unzip(self, node, key):
        if not node: return None, None
        if node.key < key:
            node.right, right = self._unzip(node.right, key)
            return node, right
        else:
            left, node.left = self._unzip(node.left, key)
            return left, node

    def insert_raw(self, key, value):
        new_node = Node(key, value)
        parent, curr = None, self.root

        while curr and new_node.priority_less(curr):
            parent = curr
            curr = curr.left if key < curr.key else curr.right
            
        l, r = self._unzip(curr, key)
        new_node.left, new_node.right = l, r
        
        if not parent: self.root = new_node
        elif key < parent.key: parent.left = new_node
        else: parent.right = new_node

    def get_all_elements(self):
        res = []
        def _walk(n):
            if not n: return
            _walk(n.left)
            res.append((n.key, n.value))
            _walk(n.right)
        _walk(self.root)
        return res

    def transpose_for_axis(self, axis_to_lead):
        new_dims = (self.dims[axis_to_lead],) + tuple(
            d for i, d in enumerate(self.dims) if i != axis_to_lead
        )
        new_tensor = ZipTreeTensor(new_dims)
        
        for idx, val in self.get_all_elements():
            new_idx = (idx[axis_to_lead],) + tuple(
                x for i, x in enumerate(idx) if i != axis_to_lead
            )
            new_tensor.insert_raw(new_idx, val)
        return new_tensor

    def find_range(self, lead_value):
        matches = []
        def _search(n):
            if not n: return
            if n.key[0] == lead_value:
                matches.append((n.key, n.value))
                _search(n.left)
                _search(n.right)
            elif n.key[0] < lead_value:
                _search(n.right)
            else:
                _search(n.left)
        _search(self.root)
        return matches

    def multiply_with_reindex(self, tensor_b, axis_a, axis_b):
        b_reindexed = tensor_b.transpose_for_axis(axis_b)
        
        res_dims = (
            tuple(d for i, d in enumerate(self.dims) if i != axis_a) +
            tuple(d for i, d in enumerate(tensor_b.dims) if i != axis_b)
        )
        result = ZipTreeTensor(res_dims)
        elements_a = self.get_all_elements()
        for idx_a, val_a in elements_a:
            k_val = idx_a[axis_a]
            matches_b = b_reindexed.find_range(k_val)
            
            for idx_b, val_b in matches_b:
                new_idx = (
                    tuple(x for i, x in enumerate(idx_a) if i != axis_a) +
                    tuple(x for i, x in enumerate(idx_b) if i != 0)
                )
                result.add_at(new_idx, val_a * val_b)
        return result
    
    def multiply(self, other, axes):
        res_dims = (
            tuple(d for i, d in enumerate(self.dims) if i != axes[0]) +
            tuple(d for i, d in enumerate(other.dims) if i != axes[1])
        )
        result = ZipTreeTensor(res_dims)
        elements_a = self.get_all_elements()
        
        for idx_a, val_a in elements_a:
            k_value = idx_a[axes[0]]
            
            matches_b = other.find_by_index_value(axes[1], k_value)
            
            for idx_b, val_b in matches_b:
                new_idx = (
                    tuple(x for i, x in enumerate(idx_a) if i != axes[0]) +
                    tuple(x for i, x in enumerate(idx_b) if i != axes[1])
                )
                result.add_at(new_idx, val_a * val_b)
                
        return result
    
    def find_by_index_value(self, axis, value):
        matches = []
        def _walk(node):
            if not node: return
            if node.key[axis] == value:
                matches.append((node.key, node.value))
            _walk(node.left)
            _walk(node.right)
        _walk(self.root)
        return matches
    
    def add_at(self, key, value):
        curr = self.root
        while curr:
            if curr.key == key:
                curr.value += value
                return
            curr = curr.left if key < curr.key else curr.right

        new_node = Node(key, value)
        parent, curr = None, self.root
        
        while curr and new_node.priority_less(curr):
            parent = curr
            curr = curr.left if key < curr.key else curr.right
            
        l, r = self._unzip(curr, key)
        new_node.left, new_node.right = l, r
        
        if not parent: self.root = new_node
        elif key < parent.key: parent.left = new_node
        else: parent.right = new_node


In [2]:
import numpy as np
import time

def compare_with_numpy(dims_a, dims_b, axis_a, axis_b, density=0.01):
    print(f"Сравнение для матриц {dims_a} и {dims_b}, плотность {density}")

    size_a = np.prod(dims_a)
    size_b = np.prod(dims_b)

    zt_a = ZipTreeTensor(dims_a)
    for _ in range(int(size_a * density)):
        idx = tuple(random.randint(0, d - 1) for d in dims_a)
        zt_a.add_at(idx, random.random())

    zt_b = ZipTreeTensor(dims_b)
    for _ in range(int(size_b * density)):
        idx = tuple(random.randint(0, d - 1) for d in dims_b)
        zt_b.add_at(idx, random.random())

    np_a = np.zeros(dims_a)
    for k, v in zt_a.get_all_elements():
        np_a[k] = v
        
    np_b = np.zeros(dims_b)
    for k, v in zt_b.get_all_elements():
        np_b[k] = v

    start_zt = time.time()
    res_zt = zt_a.multiply(zt_b, (axis_a, axis_b))
    end_zt = time.time()

    start_np = time.time()
    res_np = np.tensordot(np_a, np_b, axes=(axis_a, axis_b))
    end_np = time.time()

    zt_elements = res_zt.get_all_elements()
    diffs = []
    for k, v in zt_elements:
        diffs.append(abs(v - res_np[k]))
    
    avg_diff = np.mean(diffs) if diffs else 0

    print(f"ZipTree время: {end_zt - start_zt:.4f} сек (найдено {len(zt_elements)} элементов)")
    print(f"Numpy время:   {end_np - start_np:.4f} сек")
    print(f"Средняя разница значений: {avg_diff:.2e}")

compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.005)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.005
ZipTree время: 0.0062 сек (найдено 208 элементов)
Numpy время:   0.0136 сек
Средняя разница значений: 5.34e-19


In [3]:
import numpy as np
import time

def compare_with_numpy_reindex(dims_a, dims_b, axis_a, axis_b, density=0.01):
    print(f"Сравнение для матриц {dims_a} и {dims_b}, плотность {density}")

    size_a = np.prod(dims_a)
    size_b = np.prod(dims_b)

    zt_a = ZipTreeTensor(dims_a)
    for _ in range(int(size_a * density)):
        idx = tuple(random.randint(0, d - 1) for d in dims_a)
        zt_a.add_at(idx, random.random())

    zt_b = ZipTreeTensor(dims_b)
    for _ in range(int(size_b * density)):
        idx = tuple(random.randint(0, d - 1) for d in dims_b)
        zt_b.add_at(idx, random.random())

    np_a = np.zeros(dims_a)
    for k, v in zt_a.get_all_elements():
        np_a[k] = v
        
    np_b = np.zeros(dims_b)
    for k, v in zt_b.get_all_elements():
        np_b[k] = v

    start_zt = time.time()
    res_zt = zt_a.multiply_with_reindex(zt_b, axis_a, axis_b)
    end_zt = time.time()

    start_np = time.time()
    res_np = np.tensordot(np_a, np_b, axes=(axis_a, axis_b))
    end_np = time.time()

    zt_elements = res_zt.get_all_elements()
    diffs = []
    for k, v in zt_elements:
        diffs.append(abs(v - res_np[k]))
    
    avg_diff = np.mean(diffs) if diffs else 0

    print(f"ZipTree время: {end_zt - start_zt:.4f} сек (найдено {len(zt_elements)} элементов)")
    print(f"Numpy время:   {end_np - start_np:.4f} сек")
    print(f"Средняя разница значений: {avg_diff:.2e}")


compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.005)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.005)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.005
ZipTree время: 0.0163 сек (найдено 179 элементов)
Numpy время:   0.0027 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.005
ZipTree время: 0.0020 сек (найдено 179 элементов)
Numpy время:   0.0014 сек
Средняя разница значений: 0.00e+00


In [4]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.05)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.05)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.05
ZipTree время: 0.6785 сек (найдено 15064 элементов)
Numpy время:   0.0029 сек
Средняя разница значений: 3.07e-18

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.05
ZipTree время: 0.1136 сек (найдено 15258 элементов)
Numpy время:   0.0043 сек
Средняя разница значений: 3.41e-18


In [5]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.02)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.02)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.02
ZipTree время: 0.1207 сек (найдено 2945 элементов)
Numpy время:   0.0048 сек
Средняя разница значений: 5.87e-19

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.02
ZipTree время: 0.0247 сек (найдено 3022 элементов)
Numpy время:   0.0035 сек
Средняя разница значений: 8.50e-19


In [6]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.01)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.01)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.01
ZipTree время: 0.0350 сек (найдено 781 элементов)
Numpy время:   0.0017 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.01
ZipTree время: 0.0069 сек (найдено 782 элементов)
Numpy время:   0.0011 сек
Средняя разница значений: 0.00e+00


In [7]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.005)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.005)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.005
ZipTree время: 0.0099 сек (найдено 218 элементов)
Numpy время:   0.0005 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.005
ZipTree время: 0.0029 сек (найдено 187 элементов)
Numpy время:   0.0033 сек
Средняя разница значений: 2.97e-19


In [8]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.002)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.002)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.002
ZipTree время: 0.0023 сек (найдено 31 элементов)
Numpy время:   0.0015 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.002
ZipTree время: 0.0015 сек (найдено 34 элементов)
Numpy время:   0.0022 сек
Средняя разница значений: 0.00e+00


In [9]:
compare_with_numpy((200, 200), (200, 200), 1, 0, density=0.001)
print('\nWith reindex:')
compare_with_numpy_reindex((200, 200), (200, 200), 1, 0, density=0.001)

Сравнение для матриц (200, 200) и (200, 200), плотность 0.001
ZipTree время: 0.0004 сек (найдено 9 элементов)
Numpy время:   0.0017 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (200, 200) и (200, 200), плотность 0.001
ZipTree время: 0.0006 сек (найдено 7 элементов)
Numpy время:   0.0006 сек
Средняя разница значений: 0.00e+00


Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.02  
ZipTree время: 198.4829 сек (найдено 0 элементов)  
Numpy время:   0.0346 сек  
Средняя разница значений: 0.00e+00  
  
With reindex:  
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.02  
ZipTree время: 4.5108 сек (найдено 0 элементов)  
Numpy время:   0.0336 сек  
Средняя разница значений: 0.00e+00

In [10]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.01)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.01)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.01
ZipTree время: 23.5683 сек (найдено 94054 элементов)
Numpy время:   0.0288 сек
Средняя разница значений: 1.89e-19

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.01
ZipTree время: 0.9261 сек (найдено 94440 элементов)
Numpy время:   0.0286 сек
Средняя разница значений: 1.76e-19


In [11]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.002)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.002)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.002
ZipTree время: 0.5899 сек (найдено 3991 элементов)
Numpy время:   0.0281 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.002
ZipTree время: 0.0474 сек (найдено 4027 элементов)
Numpy время:   0.0264 сек
Средняя разница значений: 0.00e+00


In [12]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.0015)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.0015)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0015
ZipTree время: 0.3780 сек (найдено 2369 элементов)
Numpy время:   0.0288 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0015
ZipTree время: 0.0318 сек (найдено 2260 элементов)
Numpy время:   0.0279 сек
Средняя разница значений: 0.00e+00


In [13]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.001)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.001)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.001
ZipTree время: 0.1646 сек (найдено 1050 элементов)
Numpy время:   0.0277 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.001
ZipTree время: 0.0149 сек (найдено 1030 элементов)
Numpy время:   0.0270 сек
Средняя разница значений: 0.00e+00


In [14]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.0005)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.0005)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0005
ZipTree время: 0.0487 сек (найдено 235 элементов)
Numpy время:   0.0268 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0005
ZipTree время: 0.0048 сек (найдено 252 элементов)
Numpy время:   0.0278 сек
Средняя разница значений: 0.00e+00


In [15]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.0004)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.0004)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0004
ZipTree время: 0.0309 сек (найдено 145 элементов)
Numpy время:   0.0303 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0004
ZipTree время: 0.0037 сек (найдено 169 элементов)
Numpy время:   0.0266 сек
Средняя разница значений: 0.00e+00


In [16]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.0003)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.0003)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0003
ZipTree время: 0.0186 сек (найдено 86 элементов)
Numpy время:   0.0286 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0003
ZipTree время: 0.0029 сек (найдено 91 элементов)
Numpy время:   0.0273 сек
Средняя разница значений: 0.00e+00


In [17]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 0, density=0.0001)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 0, density=0.0001)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0001
ZipTree время: 0.0022 сек (найдено 7 элементов)
Numpy время:   0.0272 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0001
ZipTree время: 0.0009 сек (найдено 14 элементов)
Numpy время:   0.0275 сек
Средняя разница значений: 0.00e+00


In [18]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 1, density=0.002)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 1, density=0.002)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.002
ZipTree время: 0.8126 сек (найдено 3984 элементов)
Numpy время:   0.0273 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.002
ZipTree время: 0.0439 сек (найдено 4090 элементов)
Numpy время:   0.0271 сек
Средняя разница значений: 1.36e-20


In [19]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 1, density=0.0015)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 1, density=0.0015)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0015
ZipTree время: 0.3607 сек (найдено 2157 элементов)
Numpy время:   0.0283 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.0015
ZipTree время: 0.0280 сек (найдено 2321 элементов)
Numpy время:   0.0297 сек
Средняя разница значений: 0.00e+00


In [20]:
compare_with_numpy((1000, 1000), (1000, 1000), 1, 1, density=0.001)
print('\nWith reindex:')
compare_with_numpy_reindex((1000, 1000), (1000, 1000), 1, 1, density=0.001)

Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.001
ZipTree время: 0.1682 сек (найдено 976 элементов)
Numpy время:   0.0268 сек
Средняя разница значений: 0.00e+00

With reindex:
Сравнение для матриц (1000, 1000) и (1000, 1000), плотность 0.001
ZipTree время: 0.0150 сек (найдено 1006 элементов)
Numpy время:   0.0276 сек
Средняя разница значений: 0.00e+00
